In [1]:
import os
import pickle
import gc

import polars as pl
# import pandas as pd

from pathlib import Path
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
with open('/kaggle/input/uspto-mode/metadata_pub_iloc.pkl', 'rb') as f:
    # you may want to delete this file. 
    # but size is relatively small.
    # hence, you can left the file temporarily
    meta_dict = pickle.load(f)

{k: v for k, v in list(meta_dict.items())[:5]}

{'US-1-A': 0, 'US-1-P': 1, 'US-10-A': 2, 'US-1000-A': 3, 'US-10000-A': 4}

In [3]:
patent_dir = Path('/kaggle/input/uspto-explainable-ai/patent_data')
all_titles = []
only_in_parquet = []

# Iterate over all files in the directory
for filename in tqdm(os.listdir(patent_dir)):
    if filename.endswith(".parquet"):
        file_path = patent_dir / filename
        
        df = pl.read_parquet(file_path)
        
                
        for row in df.iter_rows(named=True):
            if row['publication_number'] in meta_dict:
                all_titles.append(row['title'])
            else:
                only_in_parquet.append(row['title'])
        
        del df
        gc.collect()

print(only_in_parquet)

100%|██████████| 2251/2251 [1:05:10<00:00,  1.74s/it]

[]


In [4]:
only_in_parquet

[]

In [5]:
# # Remove Empty Documents
# # if doc != ''
# all_titles = [doc for doc in all_titles if doc]
# # the first version do closer value

# # Additional consideration `data > 1975` would matter.

In [6]:
# Step 2: Initialize the TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english')

# Step 3: Fit and transform the patent_titles to a TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(all_titles)


with open('tfidf_ti_rebuild.pkl', 'wb') as f:
    # you may want to delete this file. 
    # but size is relatively small.
    # hence, you can left the file temporarily
    pickle.dump(vectorizer, f)

In [7]:
len(all_titles)

13307658